# Lesson 01 — From an MDP to a Learned Q-table

**Driving question:** Can we create an agent that discovers a route by experience, without explicitly giving it the route?

**Time:** 60 minutes · **Dependencies:** NumPy, Matplotlib · **No RLlib/PPO yet**

In this notebook you will run an **untrained policy**, train a **tabular Q-learning policy**, and compare them fairly. You'll inspect the actual numbers that change inside the Q-table.

**Before starting, write one prediction:** What fraction of starts will the trained agent solve, compared with the untrained one?

In [ ]:
# Run this once, regardless of whether the notebook is opened in project root,
# under lessons/, or under src/lessons/.
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
module_root = next((p for p in candidates if (p / "lessons").is_dir()), None)
if module_root is None:
    repo_root = next((p for p in candidates if (p / "src" / "lessons").is_dir()), None)
    if repo_root is None:
        raise RuntimeError("Open the notebook from within your workshop repository")
    module_root = repo_root / "src"
else:
    repo_root = module_root
sys.path.insert(0, str(module_root))
OUTPUT = repo_root / "outputs" / "lesson01_notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python module root:", module_root)
print("Outputs:", OUTPUT)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
from lessons.lesson01_agent_environment.gridworld import GridWorld, ACTION_NAMES
from lessons.lesson01_agent_environment.q_learning import (
    train_agent, rollout, evaluate, evaluation_starts
)
from lessons.lesson01_agent_environment.plots import (
    plot_training, plot_before_after, plot_policy
)

## 1. The environment: the world owns the rules

In this grid, `S` is the fixed demonstration start and `G` is the goal. `#` marks an obstacle. The goal is separated from the start by a nearly continuous wall with an opening in **row 4**.

**Prediction:** Does an action `RIGHT` from `(1,2)` move the agent?

In [ ]:
env = GridWorld(seed=7)
print(env.render_ascii())
print("Number of states:", env.num_states)
print("Actions:", ACTION_NAMES)
state, info = env.reset(start=(1, 2))
next_state, reward, done, truncated, info = env.step(1)  # RIGHT
print("From (1,2) RIGHT ->", info, "reward:", reward)

## 2. The feedback loop

```mermaid
sequenceDiagram
    participant Agent
    participant World as GridWorld
    Agent->>World: action
    World-->>Agent: next state, reward, done
    Note over Agent,World: Repeat until goal or 45-step limit
```

- **State:** one integer identifying a grid cell.
- **Action:** an index for UP, RIGHT, DOWN, or LEFT.
- **Reward:** `-0.04` per step, another `-0.45` for wall bumps, `+10` for reaching the goal.
- **Episode:** ends on reaching the goal (`terminated`) or after 45 steps (`truncated`).

**Question:** Where is the rule that says the agent must move around the wall?

In [ ]:
env.reset(start=(1, 1))
for a in [1, 1, 1]:
    state, reward, done, truncated, info = env.step(a)
    print(f"action={ACTION_NAMES[a]:5s} position={info['position']} reward={reward:+.2f} bump={info['bumped']}")

## Whiteboard bridge: model the grid as a Markov Decision Process (MDP)

The agent does not need the entire history of visited cells to know what a movement command will do. In this grid, the **current cell and chosen action** are enough to determine the next cell and reward. This is the **Markov property**.

A finite MDP is often written as $\mathcal M=(\mathcal S,\mathcal A,P,R,\gamma)$:

- $\mathcal S$: state space, represented by 49 encoded cells.
- $\mathcal A$: four actions, `UP`, `RIGHT`, `DOWN`, `LEFT`.
- $P(s'\mid s,a)$: transition probabilities; deterministic in this grid.
- $R(s,a,s')$: immediate rewards (step `−0.04`, additional wall `−0.45`, additional goal `+10`).
- $\gamma$: future reward discount (`0.97` in the default training run).

**Predict:** from `(1,1)`, RIGHT moves to `(1,2)`; another RIGHT bumps into the wall and leaves the state unchanged. The environment enforces these transitions, not the Q-learning algorithm.


In [ ]:
env.reset(start=(1, 1))
for action in [1, 1]:  # RIGHT, then RIGHT again into the wall
    next_state, reward, done, truncated, info = env.step(action)
    print({'next_state': next_state, 'position': info['position'],
           'reward': reward, 'wall_bump': info['bumped']})


## Why Bellman comes next: reward *now* versus value *later*

A policy selects actions: $\pi(a\mid s)$. Q-learning seeks high **discounted return**

$$G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots.$$

Split it into *now + later*: $G_t=R_{t+1}+\gamma G_{t+1}$.

For an **optimal** action-value function, the Bellman optimality equation is

$$Q^*(s,a)=\mathbb E\left[R_{t+1}+\gamma\max_{a'}Q^*(S_{t+1},a')\mid s,a\right].$$

We do not know the expectation exactly, so each observed transition produces a **sample target**

$$y=r+\gamma\max_{a'}Q(s',a')$$

(with future term zero at the terminal goal) and Q-learning updates

$$Q(s,a)\leftarrow Q(s,a)+\alpha[y-Q(s,a)].$$

**Whiteboard calculation:** at `(1,4)`, RIGHT reaches the goal with reward `9.96`. Old Q=0, $\alpha=0.25$: new Q = `2.49`. From `(2,4)`, UP leads to `(1,4)` with reward `−0.04`; with $\gamma=0.97$, this can update another Q entry to `0.593825`. Rewards propagate backward through repeated transitions.


In [ ]:
alpha, gamma = 0.25, 0.97
q_goal = 0.0 + alpha * (9.96 - 0.0)
q_predecessor = 0.0 + alpha * (-0.04 + gamma * q_goal - 0.0)
print('Q((1,4), RIGHT) after one goal transition:', q_goal)
print('Q((2,4), UP) after one predecessor transition:', q_predecessor)


## 3. What does an untrained agent know?

Its Q-table starts at zero. `Q[state, action]` is the agent's estimate of the longer-run value of an action taken from that state. When all values tie, this lesson's code chooses the **first** action (UP). This intentionally shows the failure of arbitrary untrained behavior; other tie-breaking conventions are possible.

**Prediction:** What will repeated `UP` actions do from `(1,1)`?

In [ ]:
zero_q = np.zeros((env.num_states, env.num_actions))
print("Q table shape:", zero_q.shape)
print("Q at start:", zero_q[env.encode((1, 1))])
before = rollout(env, zero_q, start=(1, 1), seed=100)
print("Untrained outcome:", {k: v for k, v in before.items() if k != 'path'})
print(env.render_ascii(before['path']))

## 4. What counts as learning?

A programmed policy can choose actions and receive rewards **without ever changing**. In Q-learning, the agent updates the table after each transition:

$$Q(s,a) \leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

- $\alpha$: learning rate (how quickly we change our estimate)
- $\gamma$: discount factor (how much future reward matters)
- $r$: immediate feedback
- $s'$: state after the chosen action

**Try one update by hand:** Q was 2.0, reward was −0.04, best next Q is 4.0, $\alpha=0.25$, $\gamma=0.97$. Predict the new value.

In [ ]:
q_old, reward, q_next, alpha, gamma = 2.0, -0.04, 4.0, 0.25, 0.97
q_new = q_old + alpha * (reward + gamma * q_next - q_old)
print("Updated value:", q_new)
assert np.isclose(q_new, 2.46)

## 5. Train: watch experience change decisions

`train_agent()` performs repeated episodes. At first it explores frequently using **ε-greedy** actions. It updates exactly one state-action Q-value after each step.

```mermaid
flowchart TD
    S[Sample start] --> A[Choose action using ε-greedy]
    A --> E[Take step and receive reward]
    E --> U[Update Q-table]
    U --> D{Episode over?}
    D -->|No| A
    D -->|Yes| N{More episodes?}
    N -->|Yes| S
    N -->|No| T[Freeze policy and evaluate]
```

**Before executing:** Predict whether the agent's last 50 episodes should show greater success than its first 50. Be careful: training results are noisy.

In [ ]:
result = train_agent(episodes=1200, seed=7)
print("First 50 episode training success:", result.successes[:50].mean())
print("Last 50 episode training success: ", result.successes[-50:].mean())
print("Q-table entries that changed:", np.count_nonzero(result.q))
plot_training(result, OUTPUT / "learning_curve.png")
display(Image(filename=str(OUTPUT / "learning_curve.png")))

## 6. Same scenario, before vs. after

We now **freeze** the learned Q-values, select greedy actions, and evaluate. We do not keep updating the table during evaluation.

**Prediction:** Will the trained policy take a different action from `(1,1)`?

In [ ]:
after = rollout(env, result.q, start=(1, 1), seed=100)
print("Trained outcome:", {k: v for k, v in after.items() if k != 'path'})
print(env.render_ascii(after['path']))
plot_before_after(env, before, after, OUTPUT / "before_after.png")
display(Image(filename=str(OUTPUT / "before_after.png")))

## 7. Did it really improve? Use comparable metrics

Use a predetermined collection of **60 start positions** for every policy. These locations are independent of the RNG used to sample starts during training, but they are not necessarily new grid states. This tests behavior in the *same known map*, not generalization to a different map.

Compare **success rate**, **average return**, **average steps**, and **average wall bumps**. Training return and evaluation return are not interchangeable.

In [ ]:
starts = evaluation_starts(env, count=60, seed=2026)
results = {
    "untrained": evaluate(zero_q, starts=starts),
    "random": evaluate(None, starts=starts, random_policy=True),
    "trained": evaluate(result.q, starts=starts),
}
for policy, metrics in results.items():
    print(f"{policy:10s}", metrics)

## 8. Open the learned policy

Each cell's arrow shows the action corresponding to the **largest Q-value**. The route was not typed into a controller. It emerged from repeated environment transitions and learning updates.

**Question:** Why can a policy choose different actions from neighboring cells, even with the same destination?

In [ ]:
print("Q-values at demonstration start (1,1):")
state = env.encode((1, 1))
for name, val in zip(ACTION_NAMES, result.q[state]):
    print(f"  {name:5s}: {val:+.3f}")
print("Learned action:", ACTION_NAMES[np.argmax(result.q[state])])
plot_policy(env, result.q, OUTPUT / "learned_policy.png")
display(Image(filename=str(OUTPUT / "learned_policy.png")))

## 9. YOUR EXPERIMENT — fewer updates vs. more updates

Change **only** the number of training episodes; keep the seed and test starts identical.

Write your prediction here before running:

> **Prediction:** ...

Now compare Q-learning after 30 episodes and 1200 episodes. You may see seed-dependent outcomes; avoid claiming certainty from a single training run.

In [ ]:
short = train_agent(episodes=30, seed=7)
long = train_agent(episodes=1200, seed=7)
print("After 30 episodes:", evaluate(short.q, starts=starts))
print("After 1200 episodes:", evaluate(long.q, starts=starts))
# Try optional extension: compare epsilon_start=epsilon_end=0.0 over 75 episodes.

## 10. Check your understanding

Answer these in words before looking at the README:

1. Which **two lines** in `q_learning.py` actually change the learned behavior?
2. Why can a controller receive rewards without being an RL agent?
3. Why is success rate different from mean episode return?
4. What does the Q-table represent: a complete fixed route, or action values for states?
5. Why would this 7×7 experiment NOT directly control a real UAV?

**Submit:** `before_after.png` or `learning_curve.png`, one 3-policy metrics table, your advance prediction, and 3–5 sentences interpreting the result.

**Next:** In Lesson 2, we will formalize and validate a Gymnasium environment with richer observations and discrete heading actions.